In [1]:
import pandas as pd
import numpy as np

# Create a Correct Router_Benchmark

In [2]:
df = pd.read_excel('200Q-Final/Router_Benchmark_200Q-Final-1405-03-25.xlsx')
df.head()

,Source,Question_ID,Question_Text,English_Translation,Domain,Reasoning_Type,Difficulty,Reasoning_Depth,Solution_Steps,Answer
0,Olympiad1403,3,عددهای طبیعی متمایز و بزرگتر از یک و کمتر از د...,"Consider distinct natural numbers a, z, e, and...",Number Theory,Divisibility + Pairwise Coprimality / Constrai...,7,4,5,2357
1,Olympiad1403,6,اعداد حقیقی مثبت x، y، z در معادله زیر صدق کرد...,"Positive real numbers x, y, and z satisfy the ...",Algebra,Floor/Fractional-Part Equation Reasoning,8,5,6,1
2,Olympiad1403,10,زیرمجموعه S از اعداد حقیقی مثبت را «خوب» می‌گو...,A subset S of the positive real numbers is cal...,Abstract Algebra / Sets,Closure Property Reasoning,8,5,6,2
3,Olympiad1403,12,در شکل زیر داخل دایره‌ها غنچه روییده است. تعدا...,"In the figure below, buds have grown inside th...",Graph Theory / Combinatorics,Graph Propagation / Bootstrap Percolation,8,5,6,3
4,Olympiad1403,15,"چند زوج مرتب (a, b) از عددهای صحیح و مثبت پیدا...","How many ordered pairs (a, b) of positive inte...",Number Theory,Prime Factorization / Divisor Counting,7,4,5,144


In [3]:
originEnglish = df[~df['English_Translation'].isna() == False]
transEnglish = df[df['English_Translation'].isna() == False]

questions = np.concatenate([originEnglish[['Question_Text', 'Answer']], transEnglish[['English_Translation', 'Answer']]])

extracted_df = {'source':df['Source'], 'question': questions[:, 0], 'answer': questions[:, 1],
                'domain': df['Domain'],	'reasoning_type': df['Reasoning_Type'],	'difficulty': df['Difficulty'],	'reasoning_depth': df['Reasoning_Depth'], 'solution_steps':df['Solution_Steps']}

extracted_df = pd.DataFrame(extracted_df)

extracted_df

,source,question,answer,domain,reasoning_type,difficulty,reasoning_depth,solution_steps
0,Olympiad1403,The group of 10 girls should be divided into t...,462,Number Theory,Divisibility + Pairwise Coprimality / Constrai...,7,4,5
1,Olympiad1403,A ferry with a capacity of 10 people takes a g...,286,Algebra,Floor/Fractional-Part Equation Reasoning,8,5,6
2,Olympiad1403,Find the number of different perfect covers of...,9,Abstract Algebra / Sets,Closure Property Reasoning,8,5,6
3,Olympiad1403,How many integers greater than 5400 have both ...,94830,Graph Theory / Combinatorics,Graph Propagation / Bootstrap Percolation,8,5,6
4,Olympiad1403,How many sets of three integers between 1 and ...,816,Number Theory,Prime Factorization / Divisor Counting,7,4,5
...,...,...,...,...,...,...,...,...
195,Olympiad1402,"Suppose that a₁, a₂, a₃, a₄ form a geometric p...",78,Algebra / Number Theory,Geometric Progression / Rational Constraints,8,5,6
196,Olympiad1402,"The sequence (aₙ), for n∈N, is defined by a₁ =...",8,Number Theory / Recurrences,Modular Recurrence,7,4,5
197,Olympiad1402,We know that P(x) is a cubic polynomial with r...,27,Algebra,Cubic Polynomial / Double Root Constraints,8,5,6
198,Olympiad1402,"How many three-element subsets {a,b,c} of four...",140,Number Theory / Combinatorics,Modular Power Sums / Subset Counting,9,5,7


In [4]:
extracted_df.to_csv('200Q-Final/Router_Benchmark.csv', index=False)

# Step 2 Feature Processing

In [5]:
benchmark_df = pd.read_csv('200Q-Final/Router_Benchmark.csv')

## 2.1 clean difficulty & reasoning_depth

In [6]:
print('difficulty Unique values in benchmark_df')
# print(benchmark_df['difficulty'].unique())
print(benchmark_df['difficulty'].value_counts())

print()

print('reasoning_depth Unique values in benchmark_df')
# print(benchmark_df['reasoning_depth'].unique())
print(benchmark_df['reasoning_depth'].value_counts())

difficulty Unique values in benchmark_df
difficulty
Hard               52
8                  32
Medium             30
High               22
9                  14
Medium-Hard        13
7                  12
Very High          10
Very High / Top     6
Easy-Medium         6
Very Hard           1
5                   1
6                   1
Name: count, dtype: int64

reasoning_depth Unique values in benchmark_df
reasoning_depth
High           81
5              38
Medium         31
Medium-High    22
Very High      16
4              10
3               2
Name: count, dtype: int64


In [8]:
import pandas as pd

difficulty_mapping = {
    'Easy-Medium': 4,
    'Medium': 5,
    'Medium-Hard': 6,
    'High': 7,
    'Hard': 8,
    'Very High': 9,
    'Very High / Top': 10,
    'Very Hard': 10
}

reasoning_depth_mapping = {
    'Medium': 3,
    'Medium-High': 4,
    'High': 5,
    'Very High': 6
}

# Map text to numbers; keep existing numbers unchanged
benchmark_df['difficulty'] = benchmark_df['difficulty'].apply(
    lambda x: difficulty_mapping[x] if x in difficulty_mapping else x
)
benchmark_df['reasoning_depth'] = benchmark_df['reasoning_depth'].apply(
    lambda x: reasoning_depth_mapping[x] if x in reasoning_depth_mapping else x
)

# Ensure both columns are integer type
benchmark_df['difficulty'] = benchmark_df['difficulty'].astype(int)
benchmark_df['reasoning_depth'] = benchmark_df['reasoning_depth'].astype(int)

In [9]:
print('difficulty Unique values in benchmark_df')
# print(benchmark_df['difficulty'].unique())
print(benchmark_df['difficulty'].value_counts())

print()

print('reasoning_depth Unique values in benchmark_df')
# print(benchmark_df['reasoning_depth'].unique())
print(benchmark_df['reasoning_depth'].value_counts())

difficulty Unique values in benchmark_df
difficulty
8     84
7     34
5     31
9     24
6     14
10     7
4      6
Name: count, dtype: int64

reasoning_depth Unique values in benchmark_df
reasoning_depth
5    119
3     33
4     32
6     16
Name: count, dtype: int64


# 2.2 robust pre-processing function using Pandas and Scikit-Learn to build the compact feature vector.

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def process_benchmark_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Updates the benchmark dataset directly by compressing target metadata
    columns to lower dimensions, replacing them in-place, and dropping
    sparse/high-cardinality features.
    """
    # Work on a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # ---------------------------------------------------------
    # 1. Process 'domain' -> Macro Domains (One-Hot Encoded)
    # ---------------------------------------------------------
    if 'domain' in df.columns:
        def categorize_domain(d):
            d_lower = str(d).lower()
            if any(w in d_lower for w in ['combinatorics', 'probability', 'discrete', 'graph']):
                return 'Combinatorics_Discrete'
            elif any(w in d_lower for w in ['number theory']):
                return 'Number_Theory'
            elif any(w in d_lower for w in ['algebra', 'equation']):
                return 'Algebra'
            elif any(w in d_lower for w in ['geometry']):
                return 'Geometry'
            elif any(w in d_lower for w in ['olympiad', 'logic', 'game']):
                return 'Olympiad_Logic'
            else:
                return 'Other'

        # Create temporary macro domain column
        df['macro_domain'] = df['domain'].apply(categorize_domain)

        # One-hot encode
        domain_dummies = pd.get_dummies(df['macro_domain'], prefix='domain')
        expected_domains = [
            'domain_Algebra', 'domain_Combinatorics_Discrete',
            'domain_Number_Theory', 'domain_Geometry',
            'domain_Olympiad_Logic', 'domain_Other'
        ]

        # Ensure consistent columns even if a batch lacks one of the categories
        for col in expected_domains:
            if col not in domain_dummies.columns:
                domain_dummies[col] = 0

        # Append dummies and drop the original columns
        df = pd.concat([df, domain_dummies[expected_domains].astype(float)], axis=1)
        df.drop(columns=['domain', 'macro_domain'], inplace=True)

    # ---------------------------------------------------------
    # 2. Drop 'reasoning_type' (Too sparse, let embeddings handle it)
    # ---------------------------------------------------------
    if 'reasoning_type' in df.columns:
        df.drop(columns=['reasoning_type'], inplace=True)

    # ---------------------------------------------------------
    # 3. Process Ordinal Numericals (Min-Max Scaling In-Place)
    # ---------------------------------------------------------
    num_cols = [c for c in ['solution_steps', 'difficulty', 'reasoning_depth'] if c in df.columns]

    if num_cols:
        scaler = MinMaxScaler()
        for col in num_cols:
            # Ensure numeric, fill missing with median
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(df[col].median())

        # Scale in-place
        df[num_cols] = scaler.fit_transform(df[num_cols])

    # ---------------------------------------------------------
    # 4. Process Question_Length (Log1p + Min-Max Scaling In-Place)
    # ---------------------------------------------------------
    if 'Question_Length' in df.columns:
        df['Question_Length'] = pd.to_numeric(df['Question_Length'], errors='coerce')
        df['Question_Length'] = df['Question_Length'].fillna(df['Question_Length'].median())

        # Apply Log1p to handle massive token variations, then scale 0 to 1
        df['Question_Length'] = np.log1p(df['Question_Length'])

        scaler_len = MinMaxScaler()
        df[['Question_Length']] = scaler_len.fit_transform(df[['Question_Length']])

    # ---------------------------------------------------------
    # 5. Process Answer_Type (Bucket into top 3 + Other)
    # ---------------------------------------------------------
    if 'Answer_Type' in df.columns:
        # Find top 3 answer types, bucket the rest into "Other"
        top_types = df['Answer_Type'].value_counts().nlargest(3).index.tolist()
        df['Answer_Type_Cleaned'] = df['Answer_Type'].apply(lambda x: x if x in top_types else 'Other')

        # One-hot encode and drop original
        ans_dummies = pd.get_dummies(df['Answer_Type_Cleaned'], prefix='Answer_Type').astype(float)
        df = pd.concat([df, ans_dummies], axis=1)
        df.drop(columns=['Answer_Type', 'Answer_Type_Cleaned'], inplace=True)

    return df

# Usage Example:
benchmark_df_updated = process_benchmark_metadata(benchmark_df)

In [11]:
benchmark_df_updated

,source,question,answer,difficulty,reasoning_depth,solution_steps,domain_Algebra,domain_Combinatorics_Discrete,domain_Number_Theory,domain_Geometry,domain_Olympiad_Logic,domain_Other
0,Olympiad1403,The group of 10 girls should be divided into t...,462,0.500000,0.333333,0.4,0.0,0.0,1.0,0.0,0.0,0.0
1,Olympiad1403,A ferry with a capacity of 10 people takes a g...,286,0.666667,0.666667,0.6,1.0,0.0,0.0,0.0,0.0,0.0
2,Olympiad1403,Find the number of different perfect covers of...,9,0.666667,0.666667,0.6,1.0,0.0,0.0,0.0,0.0,0.0
3,Olympiad1403,How many integers greater than 5400 have both ...,94830,0.666667,0.666667,0.6,0.0,1.0,0.0,0.0,0.0,0.0
4,Olympiad1403,How many sets of three integers between 1 and ...,816,0.500000,0.333333,0.4,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
195,Olympiad1402,"Suppose that a₁, a₂, a₃, a₄ form a geometric p...",78,0.666667,0.666667,0.6,0.0,0.0,1.0,0.0,0.0,0.0
196,Olympiad1402,"The sequence (aₙ), for n∈N, is defined by a₁ =...",8,0.500000,0.333333,0.4,0.0,0.0,1.0,0.0,0.0,0.0
197,Olympiad1402,We know that P(x) is a cubic polynomial with r...,27,0.666667,0.666667,0.6,1.0,0.0,0.0,0.0,0.0,0.0
198,Olympiad1402,"How many three-element subsets {a,b,c} of four...",140,0.833333,0.666667,0.8,0.0,1.0,0.0,0.0,0.0,0.0


In [12]:
benchmark_df_updated.to_csv('200Q-Final/Router_Benchmark.csv', index=False)

## Reduce the Benchmark Size

In [13]:
import pandas as pd

# Load data
df = pd.read_csv('200Q-Final/Router_Benchmark.csv')

# Split into first 49 questions (kept as-is) and the rest (to be reduced)
first_part = df.iloc[:57]
rest_part = df.iloc[57:]

# Sources that should be split into 3 (keep 1/3) instead of the default 2 (keep 1/2)
divide_by_3_sources = {'HendrycksMATH', 'PolyMath', 'OmniMath'}

def reduce_group(g):
    divisor = 3 if g.name in divide_by_3_sources else 2
    return g.iloc[:len(g) // divisor]

# For the remaining questions, group by source and keep only a fraction of each group
reduced_rest = (
    rest_part
    .groupby('source', group_keys=False)
    .apply(reduce_group)
)

# Combine the first 49 questions with the reduced remaining questions
new_df = pd.concat([first_part, reduced_rest], ignore_index=True)

print(f"Original size: {len(df)}")
print(f"First part (unchanged): {len(first_part)}")
print(f"Rest part (before reduction): {len(rest_part)}")
print(f"Rest part (after reduction): {len(reduced_rest)}")
print(f"New combined size: {len(new_df)}")
print("\nPer-source counts in reduced rest part:")
print(reduced_rest['source'].value_counts())

Original size: 200
First part (unchanged): 57
Rest part (before reduction): 143
Rest part (after reduction): 52
New combined size: 109

Per-source counts in reduced rest part:
source
HendrycksMATH    15
PolyMath         12
OmniMath         10
AIME2026          5
CombiBench        5
Olympiad1402      5
Name: count, dtype: int64


C:\Users\CMG\AppData\Local\Temp\ipykernel_16816\1832274852.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(reduce_group)


In [14]:
# Save the new dataset
new_df.to_csv('200Q-Final/Router_Benchmark_reduced.csv', index=False)
new_df.columns

Index(['source', 'question', 'answer', 'difficulty', 'reasoning_depth',
       'solution_steps', 'domain_Algebra', 'domain_Combinatorics_Discrete',
       'domain_Number_Theory', 'domain_Geometry', 'domain_Olympiad_Logic',
       'domain_Other'],
      dtype='object')